# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abcreativeakshay/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

---

**The queue I hand to a reviewer** is the reference pipeline's blended final queue
(`outputs/refresh_queue.csv`): all 30,000 pages ranked by `final_refresh_score`, which blends
the random-forest probability with the transparent baseline score. Every row carries a
`suggested_action` and plain-language reason codes (`final_reason_codes`), so nobody has to
trust a bare number — the queue always says *what* to look at and *why*.

**How the work is ordered:**

1. **Weekly batches of 50, top-down.** Review capacity is the constraint this whole project
   is framed around (ML-03), and Precision@50 is the metric that describes a batch of 50.
   On clients the model had never seen, the model behind this queue put 34 of its top 50 on
   pages carrying the observed decline label (P@50 = 0.68, ML-08) — against 12 of 50 for the
   reference hand rule (0.24) and roughly 20 of 50 for random ordering at the holdout base
   rate (0.391). In this snapshot itself, 49 of the queue's top 50 carry the label; the code
   below prints that as a *description of this snapshot*, not as a promise about next month.
2. **Confidence tiers set the order inside the queue:** high (3,576 pages) first, then
   medium (11,424), then low (15,000) only if capacity remains. Low-confidence pages are
   monitor territory, not editing territory.
3. **The action says what kind of review; the reason codes say why.** Queue-wide mix:
   `refresh_and_review_ctr` 6,655 · `refresh_and_review_engagement` 1,987 · `refresh` 8,207 ·
   `expand_and_refresh` 82 · `monitor` 13,069.

**Reason codes a reviewer will see, in plain words:**

| Code | What it means |
|---|---|
| `declining_with_demand` | Observed decline pattern on a page that still has an audience |
| `model_decline_risk` | The model ranks this page high on decline risk |
| `visible_model_opportunity` | Visible page where model and baseline agree there is something worth recovering |
| `low_ctr_visible_page` / `ctr_review_candidate` | Many impressions, low CTR (rate columns are percentages: 0.5 means 0.5%) |
| `low_engagement_visible_page` / `engagement_review_candidate` | People arrive, but engagement measures low |
| `page_one_decay_risk` | Sits on page one and shows decline signals — the most valuable positions to protect |
| `general_refresh_review` | No single dominant signal; a general refresh review |
| `thin_visible_page` / `stale_visible_page` | Thin content that is still seen / long-unupdated and still seen |

**Two ordering safeguards, both learned the hard way in earlier weeks:**

- **Per-client cap.** My ML-07 top-20 turned out to be 15/20 from one large client, and
  ML-09 found the flags variant's top 50 was 49/50 from a single client. The uncapped top 50
  of this queue is 36/50 from one client. So a weekly batch of 50 takes **at most 20 pages
  from any one client**; the rest of the batch fills from the next clients down the queue.
  The code below builds this week's batch both ways so the difference is visible.
- **Measurement check before editing** (Section 3): a page with huge impressions and
  exactly-zero clicks or engagement gets flagged `check_measurement_first`, not edited blind.

**Fallback:** if the model output is ever unavailable, the queue falls back to my ML-07
points rule (`work/outputs/baseline_action_score.csv`) — fully transparent, P@50 = 0.48 on
the same holdout. Weaker than the model, stronger than the reference rule, and every point
in it can be explained to a reviewer in one sentence.


In [1]:

# Section 1 — the queue this playbook runs on: anatomy, reason codes, this week's batch.
from pathlib import Path
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "outputs" / "refresh_queue.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

queue = pd.read_csv(ROOT / "outputs" / "refresh_queue.csv")
print(f"Final blended queue: {len(queue):,} pages ranked by final_refresh_score")
print("\nConfidence tiers:")
print(queue["confidence"].value_counts())
print("\nSuggested actions:")
print(queue["suggested_action"].value_counts())

codes = Counter()
for s in queue["final_reason_codes"].dropna():
    codes.update(s.split("|"))
print("\nReason codes across the queue:")
for code, n in codes.most_common():
    print(f"  {code:28s} {n:6,}")

print("\nTop 10 of the queue:")
print(queue.head(10)[["final_rank", "final_refresh_score", "best_model_probability",
                      "confidence", "suggested_action", "impressions_90d"]].to_string(index=False))

top50 = queue.head(50)
print(f"\nDescriptive only — in THIS snapshot, {int(top50['is_declining_label'].sum())} of the queue's top 50 "
      "carry the decline label.\nThe honest out-of-sample estimate is the client-holdout P@50 = 0.68 (ML-08).")
print("\nUncapped top 50, pages per client:")
print(top50["client_id"].value_counts())

def weekly_batch(frame, n=50, cap=20):
    """Top-down batch with a per-client cap, so one portfolio cannot fill the week."""
    picked, counts = [], {}
    for idx, row in frame.iterrows():
        c = row["client_id"]
        if counts.get(c, 0) >= cap:
            continue
        picked.append(idx)
        counts[c] = counts.get(c, 0) + 1
        if len(picked) == n:
            break
    return frame.loc[picked], counts

batch, batch_counts = weekly_batch(queue)
print(f"\nThis week's batch (cap 20 pages per client): {len(batch)} pages from {len(batch_counts)} clients")
print(batch_counts)


Final blended queue: 30,000 pages ranked by final_refresh_score

Confidence tiers:
confidence
low       15000
medium    11424
high       3576
Name: count, dtype: int64

Suggested actions:
suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82
Name: count, dtype: int64

Reason codes across the queue:
  declining_with_demand        13,152
  visible_model_opportunity    10,653
  low_ctr_visible_page          9,759
  ctr_review_candidate          9,759
  general_refresh_review        9,117
  model_decline_risk            8,459
  page_one_decay_risk           7,076
  low_engagement_visible_page   6,508
  engagement_review_candidate   6,508
  thin_visible_page                82
  stale_visible_page               17

Top 10 of the queue:
 final_rank  final_refresh_score  best_model_probability confidence       suggested_action  impression

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

---

**Who uses this:** a FlyRank editor/reviewer doing weekly content-refresh triage, and me as
the analyst who maintains the queue. It supports exactly one decision: *which existing pages
get human refresh review first this week.*

**What it is:** decision-support ranking. It orders review. It does not decide what edit to
make, and it does not estimate how much traffic an edit will recover — nothing in this data
measures post-refresh outcomes, so no causal claim is available at any point in this
playbook (the claim ladder from the honest-claims skill: observed → directional →
decision-support, and no further).

**Where it stops being valid:**

- **Pages with almost no audience.** The model's floor, measured in ML-08: its false
  negatives were pages with 1–3 impressions in 90 days. The code below shows 12.5% of the
  portfolio sits under 10 impressions. The queue can only rank what measurement can see —
  "not in the queue" never means "this page is fine."
- **Unmeasured fields.** `avg_position = 0` means *no data* (1,205 pages), and ML-06 found
  position information only partly measured in this snapshot. Any reason code resting on
  position deserves a human glance for those pages.
- **New clients.** Validation was client-holdout, so the honest claim covers portfolios
  similar to these 32 clients. A brand-new client's first queue is directional until its
  own outcomes have been observed for a cycle.
- **Time.** The starter data is one snapshot of trailing-90-day aggregates with **no
  calendar dates** (ML-04). The queue describes this snapshot. It says nothing about
  seasonality, and it goes stale as pages are edited — Section 4 is the staleness plan.
- **The label itself.** `is_declining_label` is an *observed* decline in search/engagement
  metrics. It is not a quality judgment, and nothing here "predicted Google's algorithm" —
  I modelled outcomes inside one portfolio, nothing more.


In [2]:

# Section 2 — where the queue stops being valid, in numbers.
raw = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
raw["y"] = (raw["trend_direction"] == "down").astype(int)

sizes = raw.groupby("client_id").size()
print(f"Portfolio: {len(raw):,} pages, {raw['client_id'].nunique()} pseudonymized clients "
      f"({sizes.min()}–{sizes.max()} pages each)")
print(f"Observed decline base rate: {raw['y'].mean():.3f}")

quiet = raw["impressions_90d"] < 10
print(f"\nMeasurement floor: {quiet.sum():,} pages ({quiet.mean():.1%}) have <10 impressions in 90 days;")
print(f"their observed decline rate is {raw.loc[quiet, 'y'].mean():.3f} vs "
      f"{raw.loc[~quiet, 'y'].mean():.3f} for pages with an audience.")
print(f"avg_position = 0 (no position data at all): {(raw['avg_position'] == 0).sum():,} pages")
print(f"Pages with zero clicks in 90 days: {(raw['clicks_90d'] == 0).sum():,} "
      "— a zero can mean unmeasured, not unwanted.")


Portfolio: 30,000 pages, 32 pseudonymized clients (3–7008 pages each)
Observed decline base rate: 0.542

Measurement floor: 3,746 pages (12.5%) have <10 impressions in 90 days;
their observed decline rate is 0.201 vs 0.591 for pages with an audience.
avg_position = 0 (no position data at all): 1,205 pages
Pages with zero clicks in 90 days: 13,204 — a zero can mean unmeasured, not unwanted.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

---

**Before acting on any queued page, the reviewer checks five things:**

1. **Measurement sanity.** Impressions high but clicks, sessions or engagement *exactly
   zero* → treat as "possibly unmeasured" and verify tracking before touching content.
   My ML-07 pick #3 was exactly this: 208K impressions, 0 clicks, 6 sessions. The code
   below flags this pattern across the top 100.
2. **The page itself.** Does it still match its intent? Is the content actually thin or
   outdated, or is the observed decline a measurement artefact? Note `word_count = 0` in
   this data means *missing*, not empty (ML-07 pick #2) — depth cannot be judged from a
   missing value.
3. **Freshness reality.** `days_since_last_update` is coarse in this slice (median 20 days,
   75th percentile 104 — ML-07), so staleness codes are approximate. Check the real update
   history before calling a page neglected.
4. **Client balance.** Is this week's batch dominated by one portfolio? The Section 1 cap
   is the guard; the reviewer confirms it held.
5. **Business context the data cannot see.** Campaigns, migrations, seasonal pages,
   deliberate sunsets — a human knows these, the queue does not.

**The no-go list — never automated, never done from the queue alone:**

- Auto-publishing edits, rewrites, or title changes.
- Deleting, redirecting, or noindexing a page because of its queue position — in either
  direction.
- Sending the queue to a client as a promise ("these pages will recover"). No causal
  evidence exists; the queue supports review order, nothing more.
- Acting on the low-confidence tier without the full review above.
- Using the score as a quality grade for writers or pages outside refresh triage.
- Re-ranking with `trend_direction` / `trend_pct` "just to check" — those columns are the
  label's parents. ML-05 showed adding `trend_pct` produces a perfect, meaningless
  AUC of 1.000.


In [3]:

# Section 3 — pre-review flags on the top 100: what a human must check before any edit.
top100 = queue.head(100).copy()
top100["review_flag"] = ""
top100.loc[(top100["impressions_90d"] >= 5000) & (top100["clicks_90d"] == 0),
           "review_flag"] += "check_measurement_first "
top100.loc[top100["word_count"].isna() | (top100["word_count"] == 0),
           "review_flag"] += "depth_unknown "
top100.loc[top100["avg_position"] == 0, "review_flag"] += "position_unmeasured "

flagged = top100[top100["review_flag"] != ""]
print(f"{len(flagged)} of the top 100 carry at least one pre-review flag:")
print(flagged[["final_rank", "content_id", "suggested_action", "impressions_90d",
               "clicks_90d", "word_count", "review_flag"]].to_string(index=False))
print("\nUnflagged does not mean approved — it means the standard review in the text above applies.")


4 of the top 100 carry at least one pre-review flag:
 final_rank           content_id       suggested_action  impressions_90d  clicks_90d  word_count                            review_flag
         35 content_5195668f06db refresh_and_review_ctr             6635           0         0.0 check_measurement_first depth_unknown 
         39 content_e4a481bafa74 refresh_and_review_ctr             1547           0         0.0                         depth_unknown 
         40 content_c65ee459f729 refresh_and_review_ctr             6526           0      2898.0               check_measurement_first 
         82 content_c1f7f47a401b refresh_and_review_ctr             2731           1         0.0                         depth_unknown 

Unflagged does not mean approved — it means the standard review in the text above applies.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

---

Three things go stale: the snapshot itself (pages get edited, new pages appear), the
audience patterns behind the model's top features, and the client mix. I monitor all three
against the **fingerprint** the code below prints for this snapshot.

**Monitoring signals — checked each time a new snapshot arrives:**

- **Label base rate: 0.542 in this snapshot.** A shift of more than ~5 points means the
  portfolio changed character — re-validate before repeating any P@50 claim.
- **Reviewer hit rate.** Reviewers record how many of each weekly batch of 50 actually
  warranted action. If the measured hit rate falls well below the holdout P@50 (0.68) for
  two consecutive batches, treat the queue as stale.
- **Feature drift** on the top drivers (`days_with_impressions`, `log_impressions_90d`,
  `avg_position`, `content_age_days`): medians moving more than ~25% from the fingerprint
  suggests a measurement or portfolio change, not noise.
- **Missingness pattern:** the `avg_position = 0` share (1,205 pages here). A jump means
  instrumentation changed — fix measurement before retraining anything.
- **Client mix:** new clients appearing, or one client growing to dominate the *uncapped*
  queue (the Section 1 cap hides this inside the batch, so check the raw queue too).

**Retrain triggers:** a new snapshot release; any signal above firing; or the queue being
used for a client segment unlike the original 32. Retraining means re-running the same
protocol — client holdout, seed 42, banned columns out — and comparing against this
playbook's numbers *before* replacing them. A new model replaces the current one only if it
matches or beats **P@50 = 0.68 on a fresh client holdout**; otherwise the old queue stays
and the drift gets investigated first.


In [4]:

# Section 4 — this snapshot's fingerprint: the numbers future snapshots get compared against.
fv = pd.read_csv(ROOT / "data" / "processed" / "refresh_feature_vector.csv")
fingerprint = {
    "decline_base_rate": round(float(fv["is_declining_label"].mean()), 3),
    "median_days_with_impressions": float(fv["days_with_impressions"].median()),
    "median_log_impressions_90d": round(float(fv["log_impressions_90d"].median()), 2),
    "median_avg_position": float(fv["avg_position"].median()),
    "median_content_age_days": float(fv["content_age_days"].median()),
    "avg_position_zero_pages": int((fv["avg_position"] == 0).sum()),
    "n_clients": int(fv["client_id"].nunique()),
    "top50_top_client_share_uncapped": round(float(top50["client_id"].value_counts(normalize=True).iloc[0]), 2),
}
print("Snapshot fingerprint (seed 42 snapshot, starter release):")
for k, v in fingerprint.items():
    print(f"  {k:34s} {v}")


Snapshot fingerprint (seed 42 snapshot, starter release):
  decline_base_rate                  0.542
  median_days_with_impressions       81.0
  median_log_impressions_90d         6.6
  median_avg_position                10.8
  median_content_age_days            236.0
  avg_position_zero_pages            1205
  n_clients                          32
  top50_top_client_share_uncapped    0.72


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

---

The code below writes the three artifacts my capstone paper (ML-11) builds on:

- **`work/outputs/playbook_queue_top100.csv`** — the top 100 of the queue with rank, score,
  confidence, action and reason codes. (CSVs under `work/` are gitignored by design —
  GUIDE §3 — so this is a working file, not a committed one.)
- **`work/outputs/playbook_action_mix.png`** — the two charts the paper reuses: queue by
  suggested action, and queue by confidence tier.
- **`work/outputs/playbook_summary.json`** — the committed receipt: queue counts, the batch
  rule, the holdout reference metrics, and the Section 4 fingerprint. Every number the
  paper quotes from this notebook traces back to this file or to the ML-08/ML-09 receipts.


In [5]:

# Section 5 — exports for the paper.
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

out = ROOT / "work" / "outputs"
out.mkdir(parents=True, exist_ok=True)

top100_export = queue.head(100)[["final_rank", "content_id", "client_id", "final_refresh_score",
                                 "best_model_probability", "confidence", "suggested_action",
                                 "final_reason_codes"]]
top100_export.to_csv(out / "playbook_queue_top100.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
queue["suggested_action"].value_counts().plot.barh(ax=axes[0])
axes[0].set_title("Queue by suggested action")
axes[0].invert_yaxis()
queue["confidence"].value_counts().reindex(["high", "medium", "low"]).plot.barh(ax=axes[1])
axes[1].set_title("Queue by confidence tier")
axes[1].invert_yaxis()
fig.tight_layout()
fig.savefig(out / "playbook_action_mix.png", dpi=150)
plt.close(fig)

summary = {
    "assignment": "ML-10",
    "seed": 42,
    "queue_rows": int(len(queue)),
    "confidence_counts": {k: int(v) for k, v in queue["confidence"].value_counts().items()},
    "action_counts": {k: int(v) for k, v in queue["suggested_action"].value_counts().items()},
    "batch_rule": {"batch_size": 50, "per_client_cap": 20},
    "holdout_reference_metrics": {"random_forest_P@50": 0.68, "my_rule_P@50": 0.48,
                                  "reference_rule_P@50": 0.24, "holdout_base_rate": 0.391,
                                  "full_data_base_rate": 0.542},
    "fingerprint": fingerprint,
}
(out / "playbook_summary.json").write_text(json.dumps(summary, indent=2))
print("Wrote:", out / "playbook_queue_top100.csv")
print("Wrote:", out / "playbook_action_mix.png")
print("Wrote:", out / "playbook_summary.json")
print("\nNote: CSVs under work/ are gitignored by design (GUIDE section 3) — "
      "the JSON receipt and the figure are the committed artifacts.")


Wrote: /home/hatch/workspace/flyrank-ml/work/outputs/playbook_queue_top100.csv
Wrote: /home/hatch/workspace/flyrank-ml/work/outputs/playbook_action_mix.png
Wrote: /home/hatch/workspace/flyrank-ml/work/outputs/playbook_summary.json

Note: CSVs under work/ are gitignored by design (GUIDE section 3) — the JSON receipt and the figure are the committed artifacts.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*Final box left unticked on purpose: the notebook is complete and executed in `work/notebooks/`; committing it to my repo is my own submission step, handled separately.*
